# 03 — Building the CT-Level Census Tract Profile

This notebook builds the census-tract (CT) level demographic, housing, and citizenship feature
table that feeds the rest of the `toronto_election_turnout` pipeline. It supersedes
`02_build_census_tract_profile.OLD.ipynb`, which had to fall back on an archived, cached
extraction of the Census Profile because the raw StatCan bulk file wasn't available on this
machine. That gap is now closed: `02_extract_toronto_census_tracts.ipynb` extracted the full 2021
Census Profile (2,631 characteristics x 622 Toronto CTs) directly from the raw StatCan bulk file
into `data/census/toronto-ct.csv` / `data/census/toronto-ct-small.csv` at the repo root. This
notebook reads from `toronto-ct-small.csv` as its **primary** source for Blocks 1-3 — not a
workaround standing in for a missing file, but the genuine independent extraction the old notebook
could only wish for.

**What this notebook produces:** one CT-level table (622 Toronto census tracts) with:
- **Block 1** (demographic): population, age structure, median age, household size, low income,
  education, unemployment, household/family structure, labour force detail, income, housing
  stability.
- **Block 2** (housing): tenure, dwelling structure type (detached / semi / apartment), condo
  status, dwelling vintage and condition, plus the two density variables (`apartments_per_km2`,
  `condos_per_km2`).
- **Block 3** (immigration / citizenship / diversity): citizenship, immigrant status, visible
  minority, official language knowledge, mother tongue, generation status, non-permanent residents.
- A handful of Block 5 fields that happen to live in the same Census Profile row (subsidized
  housing share, transit commute share, long-commute share, work-from-home share) — carried
  through here since they cost nothing extra, but they are Block 5's business downstream, not
  Block 2's.

**What's new versus the old notebook:** this notebook widens Blocks 1-3 substantially — new
household/family structure, education/labour/class, income, housing/stability, and
immigration/language variables that the old notebook's cached-extraction workaround never had
access to (the archived master CSV only ever curated ~35 characteristics). Every new variable is
genuinely new and has no archived ground truth to check against; existing variables are checked
for exact agreement against the archived master (Section 10) as a real independent cross-check now
that the two extraction paths are actually independent.

**One remaining cached-file exception:** `population_18plus` is still read from the same archived
cached intermediate the old notebook used, not derived from the new primary source. See the
markdown note in Section 4 for why.

**Out of scope here (per the project-level plan):** poll-to-CT interpolation, Block 4/5 joins,
median imputation, and the PLS model — those are later notebooks in this pipeline.

In [1]:
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

In [2]:
# -- Paths --------------------------------------------------------------------
REPO_ROOT = Path("../..").resolve()
DATA_ROOT = REPO_ROOT / "analysis" / "toronto_election_turnout" / "data"
ARCHIVE_ROOT = DATA_ROOT / "archive"
CENSUS_ROOT = ARCHIVE_ROOT / "census"
CENSUS_PROCESSED = CENSUS_ROOT / "processed"

# CT geometry universe (622 Toronto CTs) -- also our ct_id/ctuid/dguid crosswalk.
CT_GEOMETRY_GEOJSON = CENSUS_PROCESSED / "ct" / "statcan_2021_toronto_ct.geojson"

# Single-year age cube (Table 98-10-0024-01) -- fully reproducible, used to build age bands.
CT_AGE_ZIP = CENSUS_ROOT / "raw" / "source_downloads" / "statcan_2021_ct_age_single_year_98100024-eng.zip"
CT_AGE_ZIP_MEMBER = "98100024.csv"

# The ONE remaining cached intermediate this notebook still depends on -- see Section 4 markdown
# for why population_18plus can't be switched to the new primary source like everything else.
CT_POPULATION_18PLUS_INTERMEDIATE = CENSUS_PROCESSED / "ct" / "intermediate" / "statcan_2021_ct_population_18plus.csv"

# Old pipeline's interpolation outputs (untouched sibling tree) -- used only to flag which CTs
# fall inside the 585-CT interpolation universe, exactly as the archived script did.
INTERPOLATION_PROCESSED = ARCHIVE_ROOT / "interpolation" / "processed"

# New primary source (built by 02_extract_toronto_census_tracts.ipynb): the full 2021 Census
# Profile for Toronto CTs, long format, one row per (CT, characteristic).
CT_PROFILE_LONG = REPO_ROOT / "data" / "census" / "toronto-ct-small.csv"

# The archived, ground-truth master output of the old (pre-extraction) pipeline -- now used only
# as an independent cross-check target for the ~35 characteristics it happens to cover, not as a
# source of anything.
CT_CENSUS_VARIABLES_MASTER = CENSUS_PROCESSED / "ct" / "statcan_2021_ct_census_variables_master.csv"

# Downstream reference for the two housing-density variables, which have no home in the master
# file (see Section 8).
HOUSING_AUGMENTED_REFERENCE = (
    ARCHIVE_ROOT / "modelling" / "processed" / "spatial_models"
    / "toronto_ct_blocks_1_5_model_input_housing_augmented_median_imputed.csv"
)

# -- Outputs --------------------------------------------------------------------
OUT_ROOT = REPO_ROOT / "data" / "toronto_election_turnout"  # relocated output root (DATA_ROOT above still points at the old analysis/ tree, used only for ARCHIVE_ROOT)
OUTPUT_DIR = OUT_ROOT / "census"
OUTPUT_CSV = OUTPUT_DIR / "ct_census_profile.csv"
OUTPUT_GEOJSON = OUTPUT_DIR / "ct_census_profile.geojson"

VERIFICATION_TOLERANCE = 1e-6

# -- 2021 Census Profile characteristic IDs -> field names ---------------------
# Ported 1:1 from the old notebook/archived build script -- these ~35-42 characteristics were
# already verified against the archived master there, and are re-verified against it again in
# Section 10 below (now a genuine independent cross-check, not a self-comparison).
PROFILE_CHARACTERISTICS = {
    "population_total": 1,
    "population_density_per_km2": 6,
    "land_area_km2": 7,
    "median_age": 40,
    "structural_type_total_dwellings": 41,
    "single_detached_house_count": 42,
    "semi_detached_house_count": 43,
    "apartment_duplex_count": 45,
    "apartment_lt5_storeys_count": 46,
    "apartment_5plus_storeys_count": 47,
    "average_household_size": 57,
    "low_income_status_total": 335,
    "low_income_lim_at_count": 340,
    "low_income_lim_at_prevalence_pct_official": 345,
    "official_language_knowledge_total": 383,
    "neither_english_nor_french_count": 387,
    "mother_tongue_total": 393,
    "non_official_mother_tongue_count": 398,
    "tenure_total_households": 1414,
    "owner_households_count": 1415,
    "renter_households_count": 1416,
    "condo_status_total_dwellings": 1418,
    "condominium_dwellings_count": 1419,
    "subsidized_housing_tenant_pct_official": 1491,
    "citizenship_total": 1522,
    "canadian_citizens_18plus_count": 1525,
    "not_canadian_citizens_count": 1526,
    "immigrant_status_total": 1527,
    "immigrants_count": 1529,
    "recent_immigrants_2016_2021_count": 1536,
    "visible_minority_total": 1683,
    "visible_minority_population_count": 1684,
    "mobility_1yr_total": 1974,
    "same_address_1yr_count": 1975,
    "moved_1yr_count": 1976,
    "mobility_5yr_total": 1983,
    "same_address_5yr_count": 1984,
    "moved_5yr_count": 1985,
    "education_25_64_total": 2014,
    "bachelors_or_higher_25_64_count": 2024,
    "labour_force_total": 2223,
    "unemployment_rate_pct_official": 2230,
    "commute_mode_total": 2603,
    "public_transit_commute_count": 2607,
}

# -- New Block 1-3 characteristics (widening this notebook does beyond the old one) -------------
# These have no archived ground truth to check against -- see Section 7.
NEW_PROFILE_CHARACTERISTICS = {
    # Household / family structure
    "household_type_total": 100,
    "couple_with_children_count": 103,
    "one_parent_family_count": 105,
    "multigenerational_household_count": 106,
    "one_person_household_count": 110,
    "marital_status_15plus_total": 58,
    "married_or_common_law_count": 59,
    # "Never married" (char 67) nests under "Not married and not living common law" (66), itself a
    # sibling of "Married or living common-law" (59) under the same 15+ marital-status total (58) --
    # confirmed via the raw CSV, same denominator married_or_common_law_share already uses.
    "never_married_count": 67,
    # Education / labour / class
    "college_or_trades_25_64_count": 2018,
    "labour_force_participation_rate_pct_official": 2228,
    "class_of_worker_total": 2239,
    "self_employed_count": 2245,
    "work_activity_worked_total": 2233,
    "full_time_year_round_work_count": 2234,
    "occupation_total": 2248,
    "occ_management_count": 2249,
    "occ_sales_service_count": 2255,
    "occ_trades_transport_count": 2256,
    "occ_manufacturing_count": 2258,
    # 25-64-restricted equivalents of chars 1999/2000 (which are 15+-only) -- confirmed via the raw
    # CSV at ids 2015/2016, under the same education_25_64_total (2014) college/bachelors already use.
    "education_25_64_none_count": 2015,
    "education_25_64_secondary_count": 2016,
    "employment_rate_pct_official": 2229,
    "not_in_labour_force_count": 2227,
    "employee_count": 2240,
    "temporary_position_count": 2242,
    "occ_business_finance_admin_count": 2250,
    "occ_natural_applied_sciences_count": 2251,
    "occ_education_law_social_gov_count": 2253,
    "occ_arts_recreation_sport_count": 2254,
    # Income
    "median_household_income": 243,
    # Housing / stability
    "period_of_construction_total": 1440,
    "dwelling_vintage_pre1960_count": 1441,
    "dwelling_condition_total": 1449,
    "dwelling_major_repairs_needed_count": 1451,
    "core_housing_need_tenant_pct_official": 1493,
    "dwelling_vintage_1961_1980_count": 1442,
    "dwelling_vintage_1981_1990_count": 1443,
    "shelter_cost_total": 1465,
    "shelter_cost_30plus_count": 1467,
    # Combined owner+tenant core-need total/count (1479/1480) -- distinct from the tenant-specific
    # core_housing_need_tenant_pct_official (1493) already above; both are kept.
    "core_housing_need_total": 1479,
    "core_housing_need_count": 1480,
    # Immigration / citizenship / language
    "non_permanent_resident_count": 1537,
    "generation_status_total": 1665,
    "second_generation_count": 1667,
    "first_generation_count": 1666,
    "religion_total": 1949,
    "no_religion_count": 1973,
    "language_at_home_total": 724,
    "non_official_language_at_home_count": 727,
    # Block 5 stragglers (computed here for free, same rationale as the three existing ones)
    "commuting_duration_total": 2611,
    "long_commute_60min_count": 2616,
    "place_of_work_status_total": 2593,
    "work_from_home_count": 2594,
}

ALL_PROFILE_CHARACTERISTICS = {**PROFILE_CHARACTERISTICS, **NEW_PROFILE_CHARACTERISTICS}
print(f"{len(PROFILE_CHARACTERISTICS)} ported + {len(NEW_PROFILE_CHARACTERISTICS)} new = "
      f"{len(ALL_PROFILE_CHARACTERISTICS)} total raw characteristics to pivot")

44 ported + 53 new = 97 total raw characteristics to pivot


## 1. CT geometry universe

The stored Toronto CT boundary file (`statcan_2021_toronto_ct.geojson`) is the master list of the
622 census tracts that make up the City of Toronto, and gives us `ct_id`/`ctuid`/`dguid`. Every
other source in this notebook gets filtered down to this set of CTs before anything else happens.
`contains_toronto_da` flags the (rare) CT that intersects the Toronto boundary but has no verified
Toronto dissemination area inside it -- those CTs are excluded from the DA-weighted interpolation
universe downstream (see Section 9), but they still get a full demographic profile here.

*Ported verbatim from the old notebook.*

In [3]:
ct_gdf = gpd.read_file(CT_GEOMETRY_GEOJSON)
geometry_df = ct_gdf[["geo_id", "CTUID", "DGUID", "CTNAME", "contains_toronto_da", "geometry"]].rename(
    columns={"geo_id": "ct_id", "CTUID": "ctuid", "DGUID": "dguid", "CTNAME": "geo_name"}
)
geometry_df["ct_id"] = geometry_df["ct_id"].astype(float)

print(f"{len(geometry_df)} CTs total; {geometry_df['contains_toronto_da'].sum()} contain a Toronto DA")
geometry_df.head(3)

622 CTs total; 585 contain a Toronto DA


,ct_id,ctuid,dguid,geo_name,contains_toronto_da,geometry
0,5350528.35,5350528.35,2021S05075350528.35,0528.35,False,"MULTILINESTRING ((-79.60047 43.64379, -79.6005..."
1,5350530.02,5350530.02,2021S05075350530.02,0530.02,False,"MULTILINESTRING ((-79.62218 43.72238, -79.6223..."
2,5350411.08,5350411.08,2021S05075350411.08,0411.08,False,"MULTILINESTRING ((-79.53482 43.77269, -79.5347..."


## 2. Census Profile characteristics (Blocks 1-3) -- the new primary source

`toronto-ct-small.csv` is long format: one row per (CT, characteristic), with `GEO_NAME` holding
the CT id and `CHARACTERISTIC_ID` the StatCan characteristic code. `C1_COUNT_TOTAL` holds the value
for *every* characteristic type here -- raw counts, pre-computed rates/percentages, and dollar
medians alike (confirmed by spot-checking several rate-type characteristics against the archived
master, which was itself built by reading the exact same StatCan field under a different extraction
path). So pivoting this one column on `CHARACTERISTIC_ID` is all that's needed to get every raw
characteristic we want, ported and new alike, in one pass.

In [4]:
long_df = pd.read_csv(CT_PROFILE_LONG)
long_df["ct_id"] = long_df["GEO_NAME"].astype(float)

have_ids = set(long_df["CHARACTERISTIC_ID"].unique())
missing_ids = {name: cid for name, cid in ALL_PROFILE_CHARACTERISTICS.items() if cid not in have_ids}
assert not missing_ids, f"Characteristic IDs missing from toronto-ct-small.csv: {missing_ids}"

wanted_ids = set(ALL_PROFILE_CHARACTERISTICS.values())
id_to_name = {cid: name for name, cid in ALL_PROFILE_CHARACTERISTICS.items()}

profile_wide = (
    long_df[long_df["CHARACTERISTIC_ID"].isin(wanted_ids)]
    .pivot(index="ct_id", columns="CHARACTERISTIC_ID", values="C1_COUNT_TOTAL")
    .rename(columns=id_to_name)
    .reset_index()
)

profile_cols = list(ALL_PROFILE_CHARACTERISTICS.keys())
profile_raw = profile_wide[["ct_id"] + profile_cols].copy()

print(f"Pivoted {profile_raw.shape[1] - 1} raw Block 1-3 characteristics for {len(profile_raw)} CTs")
profile_raw.head(3)

Pivoted 97 raw Block 1-3 characteristics for 622 CTs


CHARACTERISTIC_ID,ct_id,population_total,population_density_per_km2,land_area_km2,median_age,structural_type_total_dwellings,single_detached_house_count,semi_detached_house_count,apartment_duplex_count,apartment_lt5_storeys_count,...,second_generation_count,first_generation_count,religion_total,no_religion_count,language_at_home_total,non_official_language_at_home_count,commuting_duration_total,long_commute_60min_count,place_of_work_status_total,work_from_home_count
0,5350001.0,599.0,87.8,6.82,40.4,235.0,15.0,85.0,5.0,40.0,...,200.0,160.0,590.0,360.0,600.0,25.0,180.0,0.0,375.0,185.0
1,5350002.0,604.0,178.0,3.39,62.0,285.0,250.0,25.0,5.0,10.0,...,140.0,150.0,600.0,375.0,605.0,5.0,115.0,10.0,245.0,125.0
2,5350003.0,457.0,483.3,0.95,38.0,265.0,0.0,0.0,0.0,0.0,...,160.0,255.0,520.0,200.0,455.0,55.0,195.0,20.0,330.0,135.0


**Takeaway:** all characteristic IDs we asked for (ported + new groupings/denominators listed in
Section 7's variables) are present in the new extraction and pivot cleanly to one row per CT, with
no missing CTs. `canadian_citizens_18plus_count` (characteristic 1525) now comes from this same
pivot -- the old notebook's cached `statcan_2021_ct_citizens_18plus.csv` intermediate is no longer
needed, since this value is confirmed (separately, outside this notebook) to match that cached file
exactly (max diff 0.0 across all 622 CTs).

## 3. Age bands from the single-year age cube (fully reproducible from raw)

The single-year age cube (`statcan_2021_ct_age_single_year_98100024-eng.zip`, Table 98-10-0024-01,
**100% Census data**) is built from scratch here, same as before: sum single years 5-17 for school
age, 18-29 and 30-64 for the two adult age bands, and take the official published "65 years and
over" aggregate rather than summing single years past 64 (StatCan doesn't publish single years
above 64 in this table). `population_age_total` is the official "Total - Age" row, which includes
institutional residents -- it will differ slightly from the Census Profile's `population_total`
(100% vs. published/rounded figures), a difference the archived script's own QA report calls out as
expected StatCan rounding.

*The 18-29 / 30-64 split (previously 18-34 / 35-64) is this notebook's own re-binning; everything
else here is ported verbatim from the old notebook.*

In [5]:
with zipfile.ZipFile(CT_AGE_ZIP) as archive:
    with archive.open(CT_AGE_ZIP_MEMBER) as raw:
        age_df = pd.read_csv(
            raw,
            encoding="utf-8-sig",
            usecols=[
                "DGUID",
                "Age (in single years), average age and median age (128)",
                "Gender (3):Total - Gender[1]",
            ],
        )
age_df.columns = ["dguid", "age_label", "count"]

target_dguids = set(geometry_df["dguid"])
age_df = age_df[age_df["dguid"].isin(target_dguids)].copy()
print(f"{len(age_df)} age-cube rows kept for the {len(target_dguids)} target CTs")


def classify_age_band(age_label: str) -> str | None:
    if age_label == "Total - Age":
        return "population_age_total"
    if age_label == "65 years and over":
        return "age_65_plus_count"
    if age_label.isdigit():
        age_value = int(age_label)
        if 5 <= age_value <= 17:
            return "school_age_5_17_count"
        if 18 <= age_value <= 29:
            return "age_18_29_count"
        if 30 <= age_value <= 64:
            return "age_30_64_count"
    return None


age_df["band"] = age_df["age_label"].map(classify_age_band)

age_bands = (
    age_df.dropna(subset=["band"])
    .groupby(["dguid", "band"])["count"]
    .sum()
    .unstack("band")
    .reindex(columns=["population_age_total", "school_age_5_17_count", "age_18_29_count", "age_30_64_count", "age_65_plus_count"])
    .reset_index()
)
for col in ["school_age_5_17_count", "age_18_29_count", "age_30_64_count"]:
    age_bands[col] = age_bands[col].fillna(0)

age_bands = age_bands.merge(geometry_df[["ct_id", "dguid"]], on="dguid", how="inner").drop(columns="dguid")
print(f"Age bands built for {len(age_bands)} CTs")
age_bands.head(3)

79616 age-cube rows kept for the 622 target CTs
Age bands built for 622 CTs


,population_age_total,school_age_5_17_count,age_18_29_count,age_30_64_count,age_65_plus_count,ct_id
0,600.0,70.0,80.0,355.0,55.0,5350001.0
1,605.0,40.0,60.0,220.0,250.0,5350002.0
2,460.0,25.0,60.0,285.0,45.0,5350003.0


## 4. Population 18+ (the one remaining cached intermediate)

`population_18plus` still comes from the cached intermediate CSV
(`statcan_2021_ct_population_18plus.csv`), not from the new primary source. This was checked
explicitly: summing the single-year age-cube bands built in Section 3
(`age_18_29_count + age_30_64_count + age_65_plus_count`) does **not** match this cached file --
there's a systematic ~55-155 person gap per CT, always positive (the cached intermediate is always
larger), not floating-point noise. Since the two don't agree and the age-cube sum isn't obviously
"more correct" than the cached file's own purpose-built extraction, this notebook keeps reading
`population_18plus` from the same cached intermediate the old notebook used, rather than silently
switching to a value that's verifiably different. Its `value_status` column is carried through so a
downstream reader can tell a genuine zero from a StatCan confidentiality suppression.

This is the **only** cached-file dependency left in this notebook -- citizenship (Section 2) no
longer needs one.

In [6]:
population_df = pd.read_csv(CT_POPULATION_18PLUS_INTERMEDIATE)
population_df["ct_id"] = population_df["geo_id"].astype(float)
population_df = population_df.rename(columns={"value_status": "population_18plus_status"})[
    ["ct_id", "population_18plus", "population_18plus_status"]
]

print(population_df.shape)
population_df.head(3)

(622, 3)


,ct_id,population_18plus,population_18plus_status
0,5350001.0,480,published
1,5350002.0,555,published
2,5350003.0,410,published


## 5. Assemble the raw variable table

Join geometry, the pivoted Block 1-3 characteristics (ported + new), the freshly-built age bands,
and the one remaining cached 18+ intermediate, all on `ct_id`.

In [7]:
raw_df = (
    geometry_df.drop(columns="geometry")
    .merge(profile_raw, on="ct_id", how="left")
    .merge(age_bands, on="ct_id", how="left")
    .merge(population_df, on="ct_id", how="left")
)

print(raw_df.shape)
raw_df.head(3)

(622, 109)


,ct_id,ctuid,dguid,geo_name,contains_toronto_da,population_total,population_density_per_km2,land_area_km2,median_age,structural_type_total_dwellings,...,long_commute_60min_count,place_of_work_status_total,work_from_home_count,population_age_total,school_age_5_17_count,age_18_29_count,age_30_64_count,age_65_plus_count,population_18plus,population_18plus_status
0,5350528.35,5350528.35,2021S05075350528.35,0528.35,False,3056.0,203.3,15.03,43.6,880.0,...,90.0,1495.0,420.0,3060.0,430.0,540.0,1470.0,470.0,2525,published
1,5350530.02,5350530.02,2021S05075350530.02,0530.02,False,3117.0,7798.3,0.40,35.2,1000.0,...,140.0,1445.0,135.0,3120.0,410.0,710.0,1295.0,470.0,2510,published
2,5350411.08,5350411.08,2021S05075350411.08,0411.08,False,7466.0,252.7,29.54,31.2,3600.0,...,345.0,4175.0,1365.0,7465.0,615.0,2380.0,3550.0,495.0,6470,published


## 6. Derived shares -- Blocks 1, 2, 3 (ported, verified)

All ~25 derived variables below are ported directly from the old notebook's `add_derived()` logic,
unchanged formula-for-formula -- only the raw characteristics feeding them now come from the new
pivot instead of the archived master CSV. A `denominator == 0` (or missing) always yields a missing
share, never a divide-by-zero.

A quick note on *why* some shares use a 25%-sample denominator and others use the 100%-sample age
cube: **tenure, condo status, mobility, and citizenship** are long-form (25%-sample) Census Profile
characteristics, so their shares use the matching 25%-sample total as denominator. **Age bands**
come from the 100%-sample single-year age table. Mixing a 25%-sample numerator with a 100%-sample
denominator (as `citizen_adult_share` does -- 25%-sample citizens over the 100%-sample
`population_18plus`) is why that one share can occasionally exceed 1 for a single CT; this is not a
bug.

Three of these (`subsidized_housing_tenant_share`, `transit_commute_share`,
`school_age_5_17_share`) are Block 5 variables by the project's block taxonomy, not Block 2 --
they're computed here only because they live in the same Census Profile row and cost nothing extra;
a downstream notebook decides which block each column belongs to when it builds the final feature
table.

In [8]:
def safe_divide(numerator, denominator):
    '''None/NaN whenever the denominator is missing or zero -- never a divide-by-zero share.'''
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = numerator / denominator
    result[denominator == 0] = np.nan
    return result


df = raw_df.copy()

# --- Block 1: demographic shares ---
df["age_18_29_share"] = safe_divide(df["age_18_29_count"], df["population_age_total"])
df["age_30_64_share"] = safe_divide(df["age_30_64_count"], df["population_age_total"])
df["age_65_plus_share"] = safe_divide(df["age_65_plus_count"], df["population_age_total"])
df["bachelors_or_higher_25_64_share"] = safe_divide(df["bachelors_or_higher_25_64_count"], df["education_25_64_total"])
df["low_income_lim_at_share"] = safe_divide(df["low_income_lim_at_count"], df["low_income_status_total"])
df["unemployment_rate_share"] = df["unemployment_rate_pct_official"] / 100

# --- Block 2: housing / tenure / dwelling structure shares ---
df["renter_share"] = safe_divide(df["renter_households_count"], df["tenure_total_households"])
df["owner_share"] = safe_divide(df["owner_households_count"], df["tenure_total_households"])
df["same_address_1yr_share"] = safe_divide(df["same_address_1yr_count"], df["mobility_1yr_total"])
df["same_address_5yr_share"] = safe_divide(df["same_address_5yr_count"], df["mobility_5yr_total"])
df["condo_share"] = safe_divide(df["condominium_dwellings_count"], df["condo_status_total_dwellings"])

apartment_cols = df[["apartment_duplex_count", "apartment_lt5_storeys_count", "apartment_5plus_storeys_count"]].fillna(0)
df["apartment_total_count"] = apartment_cols.sum(axis=1)
df["apartment_share"] = safe_divide(df["apartment_total_count"], df["structural_type_total_dwellings"])
df["detached_share"] = safe_divide(df["single_detached_house_count"], df["structural_type_total_dwellings"])
df["semi_detached_share"] = safe_divide(df["semi_detached_house_count"], df["structural_type_total_dwellings"])

# --- Block 3: immigration / citizenship / diversity shares ---
df["immigrant_share"] = safe_divide(df["immigrants_count"], df["immigrant_status_total"])
df["recent_immigrant_share"] = safe_divide(df["recent_immigrants_2016_2021_count"], df["immigrant_status_total"])
df["non_citizen_share"] = safe_divide(df["not_canadian_citizens_count"], df["citizenship_total"])
df["citizen_adult_share"] = safe_divide(df["canadian_citizens_18plus_count"], df["population_18plus"])
df["visible_minority_share"] = safe_divide(df["visible_minority_population_count"], df["visible_minority_total"])
# Backward-compatible alias: notebook 04 (interpolation) reads this exact column name off
# ct_census_profile.geojson as the CT-level counterpart of its DA-weighting field. It is numerically
# identical to canadian_citizens_18plus_count (same StatCan characteristic 1525, confirmed exact match).
df["citizen_canadian_18over"] = df["canadian_citizens_18plus_count"]
df["english_french_knowledge_count"] = df["official_language_knowledge_total"] - df["neither_english_nor_french_count"]
df["english_french_knowledge_share"] = safe_divide(df["english_french_knowledge_count"], df["official_language_knowledge_total"])
df["non_official_mother_tongue_share"] = safe_divide(df["non_official_mother_tongue_count"], df["mother_tongue_total"])

# --- Block 5 stragglers, computed here for free ---
df["subsidized_housing_tenant_share"] = df["subsidized_housing_tenant_pct_official"] / 100
df["transit_commute_share"] = safe_divide(df["public_transit_commute_count"], df["commute_mode_total"])
df["school_age_5_17_share"] = safe_divide(df["school_age_5_17_count"], df["population_age_total"])

print(f"{df.shape[1]} columns after adding ported derived shares")

136 columns after adding ported derived shares


## 7. New derived shares -- widening Blocks 1-3 (new, unverified)

Everything below is genuinely new: household/family structure, education/labour/class detail,
income, housing vintage/condition, and immigration/language variables the old notebook's archived
master CSV never covered. There is no archived ground truth to check these against, so instead of a
PASS/FAIL table (Section 10 is reserved for that), Section 7a below just `.describe()`s them and
sanity-checks that shares land in [0, 1] and the income/dollar figures look like plausible Toronto
household incomes.

Same `safe_divide()` pattern throughout: missing or zero denominator -> NaN, never a
divide-by-zero.

In [9]:
# --- Household / family structure ---
df["couple_with_children_share"] = safe_divide(df["couple_with_children_count"], df["household_type_total"])
df["one_parent_family_share"] = safe_divide(df["one_parent_family_count"], df["household_type_total"])
df["multigenerational_household_share"] = safe_divide(df["multigenerational_household_count"], df["household_type_total"])
df["one_person_household_share"] = safe_divide(df["one_person_household_count"], df["household_type_total"])
df["married_or_common_law_share"] = safe_divide(df["married_or_common_law_count"], df["marital_status_15plus_total"])
# Same marital-status-15+ denominator as married_or_common_law_share above.
df["never_married_share"] = safe_divide(df["never_married_count"], df["marital_status_15plus_total"])

# --- Education / labour / class ---
df["college_or_trades_25_64_share"] = safe_divide(df["college_or_trades_25_64_count"], df["education_25_64_total"])
df["labour_force_participation_rate"] = df["labour_force_participation_rate_pct_official"] / 100
df["self_employment_share"] = safe_divide(df["self_employed_count"], df["class_of_worker_total"])
df["full_time_year_round_work_share"] = safe_divide(df["full_time_year_round_work_count"], df["work_activity_worked_total"])
df["occ_management_share"] = safe_divide(df["occ_management_count"], df["occupation_total"])
df["occ_sales_service_share"] = safe_divide(df["occ_sales_service_count"], df["occupation_total"])
df["occ_trades_transport_share"] = safe_divide(df["occ_trades_transport_count"], df["occupation_total"])
df["occ_manufacturing_share"] = safe_divide(df["occ_manufacturing_count"], df["occupation_total"])
# Same occupation_total (char 2248) denominator as the four occupation shares above.
df["occ_business_finance_admin_share"] = safe_divide(df["occ_business_finance_admin_count"], df["occupation_total"])
df["occ_natural_applied_sciences_share"] = safe_divide(df["occ_natural_applied_sciences_count"], df["occupation_total"])
df["occ_education_law_social_gov_share"] = safe_divide(df["occ_education_law_social_gov_count"], df["occupation_total"])
df["occ_arts_recreation_sport_share"] = safe_divide(df["occ_arts_recreation_sport_count"], df["occupation_total"])
# Restricted to the same 25-64 population as college_or_trades_25_64_share / bachelors_or_higher_25_64_share
# (chars 2015/2016, not the 15+-only 1999/2000), same education_25_64_total denominator.
df["high_school_or_less_count"] = df[["education_25_64_none_count", "education_25_64_secondary_count"]].sum(axis=1)
df["high_school_or_less_share"] = safe_divide(df["high_school_or_less_count"], df["education_25_64_total"])
# StatCan's own precomputed rate (char 2229), read the same way unemployment_rate_share reads char 2230.
df["employment_rate_share"] = df["employment_rate_pct_official"] / 100
# Same labour-force-status-15+ total (char 2223) already extracted as labour_force_total.
df["not_in_labour_force_share"] = safe_divide(df["not_in_labour_force_count"], df["labour_force_total"])
df["temporary_worker_share"] = safe_divide(df["temporary_position_count"], df["employee_count"])

# --- Income (read directly, no denominator) ---
# median_household_income is already in df via the raw characteristic pivot (char 243).

# --- Housing / stability ---
df["dwelling_vintage_pre1960_share"] = safe_divide(df["dwelling_vintage_pre1960_count"], df["period_of_construction_total"])
df["dwelling_major_repairs_needed_share"] = safe_divide(df["dwelling_major_repairs_needed_count"], df["dwelling_condition_total"])
df["core_housing_need_tenant_share"] = df["core_housing_need_tenant_pct_official"] / 100
# Same period_of_construction_total denominator as dwelling_vintage_pre1960_share above.
df["dwelling_vintage_1961_1990_count"] = df[["dwelling_vintage_1961_1980_count", "dwelling_vintage_1981_1990_count"]].sum(axis=1)
df["dwelling_vintage_1961_1990_share"] = safe_divide(df["dwelling_vintage_1961_1990_count"], df["period_of_construction_total"])
df["shelter_cost_burden_30plus_share"] = safe_divide(df["shelter_cost_30plus_count"], df["shelter_cost_total"])
# Combined owner+tenant core-need share (char 1480/1479) -- a new, separate column alongside the
# existing tenant-specific core_housing_need_tenant_share; both are kept, downstream decides which to use.
df["core_housing_need_share"] = safe_divide(df["core_housing_need_count"], df["core_housing_need_total"])
# Same structural_type_total_dwellings denominator as the combined apartment_share above -- a
# narrower, separate variable, not a replacement for it.
df["high_rise_apartment_share"] = safe_divide(df["apartment_5plus_storeys_count"], df["structural_type_total_dwellings"])

# --- Immigration / citizenship / language ---
df["non_permanent_resident_share"] = safe_divide(df["non_permanent_resident_count"], df["citizenship_total"])
df["second_generation_share"] = safe_divide(df["second_generation_count"], df["generation_status_total"])
# Same generation_status_total denominator as second_generation_share above.
df["first_generation_share"] = safe_divide(df["first_generation_count"], df["generation_status_total"])
df["no_religion_share"] = safe_divide(df["no_religion_count"], df["religion_total"])
# Language spoken at home -- a different Census concept from non_official_mother_tongue_share (mother
# tongue is what was first learned at home in childhood; this is what's currently spoken at home).
df["non_official_language_at_home_share"] = safe_divide(df["non_official_language_at_home_count"], df["language_at_home_total"])
# Same official_language_knowledge_total denominator english_french_knowledge_share already uses.
df["neither_english_nor_french_share"] = safe_divide(df["neither_english_nor_french_count"], df["official_language_knowledge_total"])

# --- Mobility (1-year time base -- distinct from the existing 5-year same_address_5yr_share) ---
df["recent_mover_share"] = safe_divide(df["moved_1yr_count"], df["mobility_1yr_total"])

# --- Two more Block 5 stragglers, computed here for free ---
df["long_commute_60min_share"] = safe_divide(df["long_commute_60min_count"], df["commuting_duration_total"])
df["work_from_home_share"] = safe_divide(df["work_from_home_count"], df["place_of_work_status_total"])

print(f"{df.shape[1]} columns after adding new derived shares")

176 columns after adding new derived shares


### 7a. Sanity-checking the new variables

Nothing below has an archived ground truth to compare against -- this is purely a plausibility
check (shares should land in [0, 1] modulo the occasional >1 edge case from mixed-sample
denominators as noted in Section 6, and `median_household_income` should look like a real Toronto
CT median).

In [10]:
new_share_cols = [
    "couple_with_children_share", "one_parent_family_share", "multigenerational_household_share",
    "one_person_household_share", "married_or_common_law_share", "never_married_share",
    "college_or_trades_25_64_share", "labour_force_participation_rate", "self_employment_share",
    "full_time_year_round_work_share", "occ_management_share", "occ_sales_service_share",
    "occ_trades_transport_share", "occ_manufacturing_share",
    "occ_business_finance_admin_share", "occ_natural_applied_sciences_share",
    "occ_education_law_social_gov_share", "occ_arts_recreation_sport_share",
    "high_school_or_less_share", "employment_rate_share", "not_in_labour_force_share",
    "temporary_worker_share",
    "dwelling_vintage_pre1960_share", "dwelling_major_repairs_needed_share",
    "core_housing_need_tenant_share", "dwelling_vintage_1961_1990_share",
    "shelter_cost_burden_30plus_share", "core_housing_need_share", "high_rise_apartment_share",
    "non_permanent_resident_share", "second_generation_share", "first_generation_share",
    "no_religion_share", "non_official_language_at_home_share", "neither_english_nor_french_share",
    "recent_mover_share",
    "long_commute_60min_share", "work_from_home_share",
]
print("New share variables -- expect roughly [0, 1]:")
print(df[new_share_cols].describe().T[["min", "mean", "max"]])

print("\nmedian_household_income -- expect a plausible Toronto CT median (tens of thousands of dollars):")
print(df["median_household_income"].describe())

New share variables -- expect roughly [0, 1]:


                                          min      mean       max
couple_with_children_share           0.000000  0.257353  0.553922
one_parent_family_share              0.015982  0.104579  0.350394
multigenerational_household_share    0.000000  0.043458  0.258621
one_person_household_share           0.045455  0.291283  0.636364
married_or_common_law_share          0.041667  0.511351  0.665198
never_married_share                  0.069767  0.343682  0.693069
college_or_trades_25_64_share        0.000000  0.234134  0.386139
labour_force_participation_rate      0.054000  0.630198  0.905000
self_employment_share                0.000000  0.169464  0.526316
full_time_year_round_work_share      0.333333  0.528824  0.701043
occ_management_share                 0.000000  0.017104  0.123894
occ_sales_service_share              0.000000  0.242769  0.371429
occ_trades_transport_share           0.000000  0.126500  0.295775
occ_manufacturing_share              0.000000  0.044551  0.500000
occ_busine

## 8. Housing density (ported)

`apartments_per_km2` and `condos_per_km2` are simple: dwelling count divided by `land_area_km2`,
already a Block 2 characteristic on this same row (characteristic 7). These carry the plain names
`apartments_per_km2` / `condos_per_km2` here; a later notebook that assembles the full Block 1-5
feature table applies the `block2_` prefix (as `block2_apartments_per_km2` /
`block2_condos_per_km2`) when it selects columns for the model input table.

*Ported verbatim from the old notebook.*

In [11]:
df["apartments_per_km2"] = safe_divide(df["apartment_total_count"], df["land_area_km2"])
df["condos_per_km2"] = safe_divide(df["condominium_dwellings_count"], df["land_area_km2"])

df[["ct_id", "apartment_total_count", "condominium_dwellings_count", "land_area_km2", "apartments_per_km2", "condos_per_km2"]].head(3)

,ct_id,apartment_total_count,condominium_dwellings_count,land_area_km2,apartments_per_km2,condos_per_km2
0,5350528.35,30.0,80.0,15.03,1.996008,5.322688
1,5350530.02,790.0,435.0,0.40,1975.000000,1087.500000
2,5350411.08,2845.0,2890.0,29.54,96.310088,97.833446


## 9. Interpolation-universe flag (ported)

`in_interpolation_universe` flags the 585 (of 622) CTs that the poll-to-CT spatial interpolation
(a later notebook) actually estimates election outcomes for -- CTs containing at least one Toronto
polling division. This is derived by reading the *existing* interpolation outputs, which remain on
disk untouched.

In [12]:
interp_ct_ids = set()
for path in sorted(INTERPOLATION_PROCESSED.glob("*_ct_estimated_results.csv")):
    interp_ct_ids |= set(pd.read_csv(path, usecols=["ct_id"])["ct_id"].astype(float))

df["in_interpolation_universe"] = df["ct_id"].isin(interp_ct_ids)
print(f"{df['in_interpolation_universe'].sum()} of {len(df)} CTs are in the interpolation universe (expect 585)")

585 of 622 CTs are in the interpolation universe (expect 585)


## 10. Save the CT census profile

Written to `analysis/toronto_election_turnout/data/census/`, as a consolidated CSV plus a matching
GeoJSON (geometry carried through in case a later notebook wants to map these variables directly).
This overwrites the old notebook's output at the same path -- intentional, since this table
supersedes it.

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(df)} rows x {df.shape[1]} columns to {OUTPUT_CSV.relative_to(REPO_ROOT)}")

ct_profile_gdf = geometry_df[["ct_id", "geometry"]].merge(df, on="ct_id", how="right")
ct_profile_gdf = gpd.GeoDataFrame(ct_profile_gdf, geometry="geometry", crs=ct_gdf.crs)
ct_profile_gdf.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
print(f"Wrote geometry-carrying copy to {OUTPUT_GEOJSON.relative_to(REPO_ROOT)}")

Wrote 622 rows x 179 columns to data/toronto_election_turnout/census/ct_census_profile.csv


Wrote geometry-carrying copy to data/toronto_election_turnout/census/ct_census_profile.geojson


## 11. Verification against the archived master (ported, now a genuine cross-check)

The archived `statcan_2021_ct_census_variables_master.csv` was the ground-truth output of the old
pipeline, built from the raw StatCan Census Profile zip on a different machine that had it. Now
that this notebook extracts the same ~35-42 characteristics independently (via
`02_extract_toronto_census_tracts.ipynb`'s pull from the raw StatCan bulk file), this comparison is
a **real** cross-check between two independent extraction paths agreeing on the same underlying
StatCan product -- not a comparison of a file against itself, as it effectively was in the old
notebook. Every numeric column shared between the two tables is compared CT-by-CT; the max absolute
difference should be ~0 (floating-point noise only). `contains_toronto_da` and
`in_interpolation_universe` are boolean flags in this notebook's output but stored as
`"true"`/`"false"` strings in the archived master, so they're checked separately as an exact match.

In [14]:
master_df = pd.read_csv(CT_CENSUS_VARIABLES_MASTER)
master_df["ct_id"] = master_df["ct_id"].astype(float)

merged = df.merge(master_df, on="ct_id", how="inner", suffixes=("", "_master"))
print(f"Matched {len(merged)} of {len(df)} CTs against the {len(master_df)}-row archived master")

exclude_from_numeric_diff = {
    "ct_id", "ctuid", "dguid", "geo_name", "census_year", "geographic_level",
    "contains_toronto_da", "in_interpolation_universe",
    "population_18plus_status", "canadian_citizens_18plus_status",
}
numeric_shared_cols = [c for c in df.columns if c in master_df.columns and c not in exclude_from_numeric_diff]

verify_rows = []
for col in numeric_shared_cols:
    new_vals = pd.to_numeric(merged[col], errors="coerce")
    master_vals = pd.to_numeric(merged[f"{col}_master"], errors="coerce")
    diff = (new_vals - master_vals).abs()
    verify_rows.append({"variable": col, "max_abs_diff": diff.max(), "n_compared": int(diff.notna().sum())})

verify_df = pd.DataFrame(verify_rows).sort_values("max_abs_diff", ascending=False).reset_index(drop=True)
verify_df["status"] = np.where(verify_df["max_abs_diff"] < VERIFICATION_TOLERANCE, "PASS", "FAIL")

pd.set_option("display.max_rows", None)
print(verify_df.to_string(index=False))

overall = "PASS" if (verify_df["status"] == "PASS").all() else "FAIL"
print(f"\n=== Overall numeric verification: {overall} ({len(numeric_shared_cols)} columns, tolerance {VERIFICATION_TOLERANCE:g}) ===")

for flag in ["contains_toronto_da", "in_interpolation_universe"]:
    ours = merged[flag].astype(bool)
    theirs = merged[f"{flag}_master"].astype(str).str.lower() == "true"
    mismatches = int((ours != theirs).sum())
    print(f"{flag}: {mismatches} mismatches out of {len(merged)} -> {'PASS' if mismatches == 0 else 'FAIL'}")

Matched 622 of 622 CTs against the 622-row archived master
                                 variable  max_abs_diff  n_compared status
                   recent_immigrant_share  4.997121e-11         620   PASS
                    school_age_5_17_share  4.996559e-11         622   PASS
                   same_address_1yr_share  4.995993e-11         620   PASS
                   same_address_5yr_share  4.995770e-11         620   PASS
           english_french_knowledge_share  4.993905e-11         621   PASS
                              condo_share  4.991790e-11         620   PASS
         non_official_mother_tongue_share  4.988610e-11         621   PASS
                           detached_share  4.987775e-11         620   PASS
                          immigrant_share  4.987277e-11         620   PASS
          bachelors_or_higher_25_64_share  4.987144e-11         620   PASS
                      citizen_adult_share  4.986622e-11         620   PASS
                    transit_commute_share

### 11a. Bonus check: the two housing density variables (ported)

`apartments_per_km2` and `condos_per_km2` don't exist in the master file. As a bonus check, compare
them against the old pipeline's downstream housing-augmented file's `block2_apartments_per_km2` /
`block2_condos_per_km2` columns -- small non-zero diffs are expected here since that file applies
median imputation to a handful of CTs, which this notebook intentionally does not do.

In [15]:
housing_ref = pd.read_csv(HOUSING_AUGMENTED_REFERENCE, usecols=["ct_id", "block2_apartments_per_km2", "block2_condos_per_km2"])
housing_ref["ct_id"] = housing_ref["ct_id"].astype(float)
bonus = df.merge(housing_ref, on="ct_id", how="inner")
print(f"{len(bonus)} CTs matched against the downstream housing-augmented reference")

for col, ref_col in [("apartments_per_km2", "block2_apartments_per_km2"), ("condos_per_km2", "block2_condos_per_km2")]:
    diff = (bonus[col] - bonus[ref_col]).abs()
    print(f"{col}: max abs diff = {diff.max():.6g} (n={int(diff.notna().sum())})")

585 CTs matched against the downstream housing-augmented reference
apartments_per_km2: max abs diff = 4.73684e-06 (n=585)
condos_per_km2: max abs diff = 4.88372e-06 (n=583)


## Summary

This notebook produced `analysis/toronto_election_turnout/data/census/ct_census_profile.csv`
(622 rows, one per Toronto CT) plus a geometry-carrying `ct_census_profile.geojson`.

**Ported, verified** (PASS at 1e-6 tolerance against the archived master, now a genuine
independent-extraction cross-check): all original Block 1-3 raw characteristics and their ~25
derived shares (the age bands are re-binned 18-29 / 30-64, see Section 3, so `age_18_34_share` /
`age_35_64_share` are now `age_18_29_share` / `age_30_64_share` and are no longer directly
comparable to the archived master's same-named columns), the three original Block 5 stragglers
(`subsidized_housing_tenant_share`, `transit_commute_share`, `school_age_5_17_share`), the two
housing density variables (bonus check against the downstream housing-augmented file), and the two
boolean flags (`contains_toronto_da`, `in_interpolation_universe`).

**New, unverified** (no archived ground truth exists -- sanity-checked by range only in Section
7a): household/family structure (`couple_with_children_share`, `one_parent_family_share`,
`multigenerational_household_share`, `one_person_household_share`, `married_or_common_law_share`,
`never_married_share`), education/labour/class (`college_or_trades_25_64_share`,
`labour_force_participation_rate`, `self_employment_share`, `full_time_year_round_work_share`,
`occ_management_share`, `occ_sales_service_share`, `occ_trades_transport_share`,
`occ_manufacturing_share`, `occ_business_finance_admin_share`, `occ_natural_applied_sciences_share`,
`occ_education_law_social_gov_share`, `occ_arts_recreation_sport_share`,
`high_school_or_less_share`, `employment_rate_share`, `not_in_labour_force_share`,
`temporary_worker_share`), `median_household_income`, housing/stability
(`dwelling_vintage_pre1960_share`, `dwelling_major_repairs_needed_share`,
`core_housing_need_tenant_share`, `dwelling_vintage_1961_1990_share`,
`shelter_cost_burden_30plus_share`, `core_housing_need_share`, `high_rise_apartment_share`),
immigration/language (`non_permanent_resident_share`, `second_generation_share`,
`first_generation_share`, `no_religion_share`, `non_official_language_at_home_share`,
`neither_english_nor_french_share`), mobility (`recent_mover_share`, a 1-year time base distinct
from the existing 5-year `same_address_5yr_share`), and two more Block 5 stragglers
(`long_commute_60min_share`, `work_from_home_share`).

Two of the new columns are deliberately added *alongside* an existing, conceptually different
column rather than replacing it: `core_housing_need_share` (combined owner+tenant, char 1480) sits
next to the existing tenant-specific `core_housing_need_tenant_share` (char 1493), and
`non_official_language_at_home_share` (language currently spoken at home, char 727) sits next to
the existing `non_official_mother_tongue_share` (mother tongue, char 398) -- a downstream notebook
decides which one feeds the model.

**The one remaining cached-file dependency**: `population_18plus` still comes from the archived
`statcan_2021_ct_population_18plus.csv` intermediate (Section 4) -- the age-cube sum doesn't match
it (a systematic ~55-155 person gap per CT), so this notebook keeps the cached value rather than
silently substituting a verifiably different one. `canadian_citizens_18plus_count`, by contrast, no
longer needs a cached file at all -- it's read straight from the new primary source (Section 2),
confirmed elsewhere to match the old cached intermediate exactly.